# RSNA Knee Abnormality Detection — Boilerplate

Multi-label classification of knee MRI studies. For each study, predict 12 abnormality outcomes:
ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.

This notebook loads the CSVs, explores the data, provides a DICOM volume loader, and writes a constant baseline submission.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom

# ---------- locate competition data ----------
def find_data_dir():
    for base in ["/kaggle/input/competitions", "/kaggle/input"]:
        if not os.path.isdir(base):
            continue
        for d in sorted(os.listdir(base)):
            if "knee" in d.lower():
                return os.path.join(base, d)
    return "/kaggle/input/rsna-knee-abnormality-detection"

DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

TARGET_COLS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
               "Medial OA", "Lateral OA", "PF OA", "Effusion",
               "Synovitis", "Baker's", "Contusion", "Fracture"]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_series = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("train:", train.shape)
print("test:", test.shape)
print("train_series:", train_series.shape)
print("test_series:", test_series.shape)
print("sample_sub:", sample_sub.shape)

In [ ]:
train.head()

In [ ]:
# target distribution
print("Missing values in train targets:", train[TARGET_COLS].isna().sum().sum())
print("Unique target values:", np.unique(train[TARGET_COLS].values))
train[TARGET_COLS].mean().plot(kind="bar", figsize=(10, 4), title="Positive rate per target")
plt.xticks(rotation=45, ha="right")
plt.show()

train["has_abnormality"] = (train[TARGET_COLS] > 0).any(axis=1)
print("Share of studies with >=1 abnormality:", train["has_abnormality"].mean())

In [ ]:
# series-level metadata
print("Anatomical planes:")
print(train_series["Anatomical_Plane"].value_counts())
print()
print("Fluid_Sensitive:")
print(train_series["Fluid_Sensitive"].value_counts())
print()
print("Series per study:")
print(train_series.groupby("StudyInstanceUID").size().describe())

# Extended EDA

Extra exploration: column overview, missing-value structure, target co-occurrence, series-level distributions, and abnormality counts per study.

In [ ]:
def overview(df, name):
    print(f"### {name} ({df.shape[0]} rows x {df.shape[1]} cols)")
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nunique": df.nunique(dropna=False),
        "n_missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    })
    print(info.to_string())

overview(train, "train")
print()
overview(train_series, "train_series")

In [ ]:
# --- target missing-value structure ---
print("Per-target missing (NaN) counts in train:")
print(train[TARGET_COLS].isna().sum().to_string())
print()
print("Studies with any missing target:", train[TARGET_COLS].isna().any(axis=1).sum())
print("Studies with all targets present:", train[TARGET_COLS].notna().all(axis=1).sum())
print()
print("Target value distributions:")
print(train[TARGET_COLS].stack().value_counts(dropna=False).to_string())

In [ ]:
# --- target co-occurrence ---
corr = train[TARGET_COLS].fillna(0).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Target correlation matrix")
plt.tight_layout()
plt.show()

print("Top 5 most correlated target pairs:")
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print(pairs.sort_values(ascending=False).head(5).to_string())

In [ ]:
# --- abnormalities per study ---
n_abn = (train[TARGET_COLS].fillna(0) > 0).sum(axis=1)
n_abn.value_counts().sort_index().plot(kind="bar", figsize=(8, 4),
                                       title="Number of abnormalities per study")
plt.xlabel("n abnormalities")
plt.ylabel("studies")
plt.tight_layout()
plt.show()

In [ ]:
# --- series-level breakdown ---
print("Series per study (train):")
print(train_series.groupby("StudyInstanceUID").size()
      .value_counts().sort_index().rename("n_studies").to_string())
print()
print("Plane x Fluid_Sensitive counts:")
print(pd.crosstab(train_series["Anatomical_Plane"],
                  train_series["Fluid_Sensitive"]).to_string())
print()
print("Series count per study stats:")
print(train_series.groupby("StudyInstanceUID")["SeriesInstanceUID"]
      .count().describe().to_string())

In [ ]:
def load_series_volume(study_uid, series_uid, split="train"):
    """Load all DICOM slices of a series into a (Z, H, W) float32 volume."""
    d = os.path.join(DATA_DIR, f"{split}_series", study_uid, series_uid)
    files = sorted(glob.glob(os.path.join(d, "*.dcm")))
    if not files:
        return None
    slices = [pydicom.dcmread(f) for f in files]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    arr = np.stack([s.pixel_array for s in slices]).astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
    return arr

In [ ]:
# visualize one example volume
study = train["StudyInstanceUID"].iloc[0]
ser = train_series[train_series["StudyInstanceUID"] == study].iloc[0]
vol = load_series_volume(study, ser["SeriesInstanceUID"])
print("study:", study)
print("series:", ser["SeriesInstanceUID"], "| plane:", ser["Anatomical_Plane"], "| shape:", vol.shape)

idx = np.linspace(len(vol) // 4, 3 * len(vol) // 4, 4).astype(int)
fig, axes = plt.subplots(1, len(idx), figsize=(4 * len(idx), 4))
for ax, i in zip(axes, idx):
    ax.imshow(vol[i], cmap="gray")
    ax.axis("off")
    ax.set_title(f"slice {i}")
plt.show()

## Baseline

A constant baseline predicts the training-set positive rate for every target on every test study. The evaluation metric is average AUC across the 12 targets.

In [ ]:
# constant baseline -> sample submission
mean_targets = train[TARGET_COLS].mean()
sub = test[["StudyInstanceUID"]].copy()
for c in TARGET_COLS:
    sub[c] = mean_targets[c]
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head()

In [ ]:
# cross-validation setup (group by study so volumes never leak across folds)
from sklearn.model_selection import StratifiedGroupKFold

X = train.copy()
X["has_abnormality"] = (X[TARGET_COLS] > 0).any(axis=1)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = np.zeros(len(X), dtype=int)
for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, X["has_abnormality"], groups=X["StudyInstanceUID"])):
    folds[va_idx] = fold
X["fold"] = folds
print(X["fold"].value_counts().sort_index())
X.to_csv("train_with_folds.csv", index=False)